# Project: European Air Traffic Pipeline on EUROCONTROL OPDI data (Medallion Architecture, PySpark)

## DAT535 - Data Engineering

### Thomas Canisares--Morère & Mathis Nancey - Group 08

Batch pipeline that turns EUROCONTROL OPDI flight data into clean, analysis-ready datasets using the Medallion Architecture.

- Period: June 1, 2026 -> July 28, 2026 (~5.5 GB)
- Tables: flight_list (monthly), flight_events and measurements (10-day intervals), OurAirports repositories (airports, runways)

## Part 1: Setup
### Part 1.1: Imports

In [ ]:
import os
import json
import time
import datetime
import random
import csv
import hashlib
import shutil
import urllib.error
import urllib.request
from pathlib import Path
from collections import Counter, defaultdict

import findspark
findspark.init()

from pyspark.sql import SparkSession, Row
from pyspark.sql.functions import (
    col, lit, when, count, sum as spark_sum, avg,
    min as spark_min, max as spark_max, round as spark_round,
    desc, asc, to_timestamp, to_date, hour, dayofweek,
    lower, upper, trim, concat, coalesce, explode, split,
    countDistinct, collect_list, first, 
    input_file_name, regexp_extract, current_timestamp, broadcast
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, TimestampType, BooleanType, LongType
)

print("Libraries imported successfully")


### Part 1.2: SparkSession

In [ ]:
# SparkSession
spark = SparkSession.builder \
    .appName("DAT535-Project_Thomas_Mathis") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.session.timeZone", "UTC") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print(f"Spark Version: {spark.version}")
print(f"Application ID: {spark.sparkContext.applicationId}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")


### Part 1.3: Parameters

In [ ]:
START = datetime.date(2026, 6, 1)
END   = datetime.date(2026, 7, 29)
ORIGIN     = datetime.date(2022, 1, 1)            # the 10-day periods are counted starting from this date
MIN_FREE_GB = 4.0                                 # minimum free space in GB before downloading a file
RETRIES     = 3                                   # number of download attempts before giving up
PAUSE_S     = 0.5                                 # pause between downloads in seconds

UA = {"User-Agent": "UiS-DAT535-group08"}         # user agent for HTTP requests
BASE_URL   = "https://www.eurocontrol.int/performance/data/download/OPDI/v002"
REF_URLS   = {"airports.csv": "https://davidmegginson.github.io/ourairports-data/airports.csv",
              "runways.csv":  "https://davidmegginson.github.io/ourairports-data/runways.csv"}

PARQUET_TABLES = ["flight_list", "flight_events", "measurements"]

DATA_ROOT      = Path.home() / "opdi-data"
RAW_DIR        = DATA_ROOT / "raw"                # source files as downloaded
BRONZE_DIR     = DATA_ROOT / "bronze"             # raw data + lineage columns
SILVER_DIR     = DATA_ROOT / "silver"
QUARANTINE_DIR = DATA_ROOT / "quarantine"
GOLD_DIR       = DATA_ROOT / "gold"

## Part 2: Bronze Layer
### Part 2.1: Retrieve the data
#### Part 2.1.1: utility functions

In [ ]:
# months() generates the first day of each month in [start, end[
#   - start: the start date (inclusive)
#   - end: the end date (exclusive)
def months(start, end):
    m = datetime.date(start.year, start.month, 1)
    while m < end:
        yield m
        m = datetime.date(m.year + (m.month == 12), m.month % 12 + 1, 1)

# Generate 10-day windows starting from ORIGIN
#   - start: the start date (inclusive)
#   - end: the end date (exclusive)
#   - days: the number of days in each window
def windows(start, end, days=10):
    w = ORIGIN + datetime.timedelta(days=((start - ORIGIN).days // days) * days)
    while w < end:
        yield w, w + datetime.timedelta(days=days)
        w += datetime.timedelta(days=days)

# We return here the HTTP status, the expected size in bytes, and the last modified date (if available)
#   - url: the URL to check
def head(url):
    req = urllib.request.Request(url, method="HEAD", headers=UA)
    try:
        with urllib.request.urlopen(req, timeout=30) as r:
            return r.status, int(r.headers.get("Content-Length") or 0), r.headers.get("Last-Modified", "")
    except urllib.error.HTTPError as e:
        return e.code, 0, ""
    except Exception as e:
        return type(e).__name__, 0, ""

# this function downloads the file from the given URL to the destination path.
#   - url: the URL to download from
#   - dest: the destination path (Path object)
def download(url, dest):
    tmp = dest.with_name(dest.name + ".part") # temporary file during download
    for attempt in range(1, RETRIES + 1):
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=300) as r, open(tmp, "wb") as f:
                shutil.copyfileobj(r, f, 1 << 20) # copy in 1 MB chunks
            os.replace(tmp, dest)
            return "downloaded"
        except Exception as e:
            print(f"    attempt {attempt}/{RETRIES} failed : {e}")
            tmp.unlink(missing_ok=True)
            time.sleep(5 * attempt)
    return "failed"

# Compute the SHA256 hash of a file at the given path
#   - path: the path to the file (Path object)
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

# Return the free disk space in GB for the RAW_DIR
def free_gb():
    return shutil.disk_usage(RAW_DIR).free / 1e9

# Return the current UTC time as a datetime object 
def utc_now():
    return datetime.datetime.now(datetime.timezone.utc)

#### Part 2.1.2: List of files

In [ ]:
files = []
for m in months(START, END):
    n = f"flight_list_{m:%Y%m}.parquet"
    files.append(("flight_list", n, f"{BASE_URL}/flight_list/{n}"))
for table in ("flight_events", "measurements"):
    for w, w_end in windows(START, END):
        n = f"{table}_{w:%Y%m%d}_{w_end:%Y%m%d}.parquet"
        files.append((table, n, f"{BASE_URL}/{table}/{n}"))
for n, url in REF_URLS.items():
    files.append(("reference", n, url))

#### Part 2.1.3: Download

The main loop that downloads the files and records the results in a manifest CSV file.
The manifest contains all the relevant information about each file downloaded and it contains the following columns:
- run_id: a unique identifier for this run of the script
- table: the table name (flight_list, flight_events, measurements, reference)
- file: the file name
- url: the URL from which the file was downloaded
- http_status: the HTTP status code returned by the server
- size_bytes: the size of the file in bytes
- last_modified: the last modified date of the file on the server (if available)
- result: the result of the download (downloaded, already_present, absent, empty, failed)
- sha256: the SHA256 hash of the file
- local_path: the local path of the downloaded file
- retrieved_at_utc: the UTC time when the file was retrieved

In [ ]:
RAW_DIR.mkdir(parents=True, exist_ok=True)
manifest_path = RAW_DIR / "manifest.csv"
run_id = utc_now().strftime("%Y%m%dT%H%M%SZ")
print(f"Period [{START}, {END}[ | {len(files)} files | destination {RAW_DIR} | free {free_gb():.1f} GB\n")
rows, t0 = [], time.time() # list to store the results and the start time
for i, (table, name, url) in enumerate(files, 1):
    status, size, last_modified = head(url)
    dest = RAW_DIR / table / name
    dest.parent.mkdir(parents=True, exist_ok=True)
    if status != 200:
        result = "absent"
    elif size == 0:
        result = "empty"
    elif dest.exists():
        result = "already_present"
    elif free_gb() - size / 1e9 < MIN_FREE_GB:
        print(f"STOP : insufficient free space before {name}")
        break
    else:
        result = download(url, dest)
    ok = result in ("downloaded", "already_present")
    rows.append(dict(run_id=run_id, table=table, file=name, url=url, http_status=status,
                     size_bytes=size, last_modified=last_modified, result=result,
                     sha256=sha256(dest) if ok else "", local_path=str(dest) if ok else "",
                     retrieved_at_utc=utc_now().isoformat(timespec="seconds")))
    print(f"[{i:2d}/{len(files)}] {table:13s} {name:45s} {size/1e6:8.1f} MB  {result}")
    time.sleep(PAUSE_S)

if rows: # write the manifest CSV file
    new = not manifest_path.exists()
    with open(manifest_path, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]))
        if new:
            w.writeheader()
        w.writerows(rows)

elapsed = time.time() - t0 

#### Part 2.1.4: Summary

In [ ]:
print(f"\nResults : {dict(Counter(r['result'] for r in rows))}") # print summary of results
for t in ("flight_list", "flight_events", "measurements", "reference"): # print summary of files downloaded for each table
    sel = [r for r in rows if r["table"] == t and r["local_path"]]
    print(f"  {t:13s} {len(sel):3d} files  {sum(int(r['size_bytes']) for r in sel)/1e9:6.2f} GB")
print(f"Duration {elapsed:.0f} s | free space left {free_gb():.1f} GB | manifest: {manifest_path}") # duration, free space, manifest path
for r in rows: # print any files that were not downloaded successfully
    if r["result"] not in ("downloaded", "already_present"):
        print(f"  ! {r['file']} -> {r['result']} (HTTP {r['http_status']})")

### Part 2.2: Schema Validation

OPDI files from different periods were observed with different schemas (e.g. `long` vs `decimal(20,0)`
identifiers, dates stored as `string` vs `timestamp`). Before loading anything into Silver, we check that
all the files of a same table share the same schema, and we record each file's schema fingerprint and
row count in `schema_report.csv` for traceability.

In [ ]:
schema_variants = defaultdict(lambda: defaultdict(list))    # group files according to their schema
report_rows = []                                            # one row per file, written to schema_report.csv

for table in PARQUET_TABLES:
    for path in sorted((RAW_DIR / table).glob("*.parquet")):
        df = spark.read.parquet(str(path))                  # reads only the metadata of the Parquet file
        schema = tuple((f.name, f.dataType.simpleString()) for f in df.schema.fields)
        n_rows = df.count()                                
        fingerprint = hashlib.sha256(str(schema).encode()).hexdigest()[:12]
        schema_variants[table][schema].append(path.name)
        report_rows.append(dict(table=table, file=path.name, n_rows=n_rows, n_columns=len(schema),
                                schema_fingerprint=fingerprint,
                                checked_at_utc=utc_now().isoformat(timespec="seconds")))

for table, variants in schema_variants.items():
    keys = list(variants)
    n_files = sum(len(v) for v in variants.values())
    print(f"\n=== {table}: {n_files} files | {len(keys)} schema variant(s) ===")
    for k, key in enumerate(keys, 1):
        names = variants[key]
        listed = ", ".join(names) if len(names) <= 3 else f"{names[0]} ... {names[-1]}"
        print(f"  Variant {k}: {len(names)} file(s) -> {listed}")
    if len(keys) == 1:                                       # single schema: show it
        for name, dtype in keys[0]:
            print(f"      {name:22s} {dtype}")
    else:                                                    # several schemas: show only the differences
        types = [dict(key) for key in keys]
        all_cols = list(dict.fromkeys(c for t in types for c in t))
        print("  Columns that differ between variants:")
        for c in all_cols:
            vals = [t.get(c, "-- missing --") for t in types]
            if len(set(vals)) > 1:
                print(f"      {c:22s} " + " | ".join(f"V{i + 1}: {v}" for i, v in enumerate(vals)))

# Row counts per file
print("\n=== Rows per file ===")
for r in report_rows:
    print(f"  {r['table']:13s} {r['file']:45s} {r['n_rows']:>12,} rows  schema {r['schema_fingerprint']}")

# Save the report
schema_report_path = RAW_DIR / "schema_report.csv"
with open(schema_report_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(report_rows[0]))
    w.writeheader()
    w.writerows(report_rows)
print(f"\nSchema report written to: {schema_report_path}")

### Part 2.3: Bronze tables (row-level lineage)

The source files in `raw/` are kept untouched. The Bronze tables in `bronze/<table>/` contain
exactly the same rows, columns and types, plus row-level lineage columns taken from the manifest:

| Column | Meaning |
|---|---|
| `_source_file` | name of the raw file the row comes from |
| `_source_period` | month (`YYYYMM`) or 10-day window start (`YYYYMMDD`) of the source file — also the partition key |
| `_ingestion_run_id` / `_ingested_at_utc` | download run that retrieved the file, and when |
| `_source_last_modified` | publication date of the file on the EUROCONTROL server |
| `_source_sha256` | fingerprint of the source file |
| `_bronze_run_id` / `_bronze_loaded_at_utc` | run that built this Bronze table, and when |

No cleaning or type conversion happens here. Tables are partitioned by source period, so re-running only
replaces the partitions of the current period. A reconciliation then checks that every raw row is in Bronze.

In [ ]:
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")   # a re-run only replaces the partitions it writes

# 1) File-level lineage: latest successful manifest entry for each file
latest = {}
with open(manifest_path, newline="") as f:
    for r in csv.DictReader(f):
        if r["result"] in ("downloaded", "already_present"):
            latest[r["file"]] = r                                    # chronological file: the last entry wins

lineage_df = spark.createDataFrame(
    [(name, r["run_id"], r["retrieved_at_utc"], r["last_modified"], r["sha256"]) for name, r in latest.items()],
    "_source_file string, _ingestion_run_id string, _ingested_at_utc string, "
    "_source_last_modified string, _source_sha256 string",
).withColumn("_ingested_at_utc", col("_ingested_at_utc").cast("timestamp"))

bronze_run_id = utc_now().strftime("%Y%m%dT%H%M%SZ")

# 2) Row-level lineage: each row gets the name of its source file, then the file metadata from the manifest
def add_lineage(df):
    return (df
        .withColumn("_source_file", regexp_extract(input_file_name(), r"([^/]+)$", 1))
        .withColumn("_source_period", regexp_extract(col("_source_file"), r"_(\d{6,8})(?:_\d{8})?\.parquet$", 1))
        .join(broadcast(lineage_df), "_source_file", "left")            # small table: broadcast join, no shuffle
        .withColumn("_bronze_run_id", lit(bronze_run_id))
        .withColumn("_bronze_loaded_at_utc", current_timestamp()))

# Files of the current period only (list built in 2.1.2)
in_scope = defaultdict(list)
for table, name, _ in files:
    if (RAW_DIR / table / name).exists():
        in_scope[table].append(str(RAW_DIR / table / name))

# 3) OPDI tables: original columns and types unchanged, partitioned by source period
for table in PARQUET_TABLES:
    t_start = time.time()
    (add_lineage(spark.read.parquet(*in_scope[table]))
        .write.mode("overwrite")
        .partitionBy("_source_period")
        .parquet(str(BRONZE_DIR / table)))
    print(f"{table:13s} -> {BRONZE_DIR / table}  ({time.time() - t_start:.0f} s)")

# 4) Reference tables (OurAirports CSV): every column kept as a string, exactly as in the source file
for path in in_scope["reference"]:
    table = Path(path).stem                                            # airports, runways
    raw_ref = spark.read.option("header", True).option("multiLine", True).option("escape", '"').csv(path)
    (add_lineage(raw_ref).drop("_source_period")
        .write.mode("overwrite").parquet(str(BRONZE_DIR / table)))
    print(f"{table:13s} -> {BRONZE_DIR / table}")

In [ ]:
expected = {}                                                          # raw row counts per file
with open(RAW_DIR / "schema_report.csv", newline="") as f:               # written in 2.2 (Parquet files)
    for r in csv.DictReader(f):
        expected[r["file"]] = int(r["n_rows"])
for path in in_scope["reference"]:                                     # CSV files: counted directly
    expected[Path(path).name] = (spark.read.option("header", True).option("multiLine", True)
                                 .option("escape", '"').csv(path).count())

recon = []
for table in PARQUET_TABLES + [Path(p).stem for p in in_scope["reference"]]:
    stats = (spark.read.parquet(str(BRONZE_DIR / table))
             .groupBy("_source_file")
             .agg(count("*").alias("n_rows"),
                  spark_sum(col("_source_sha256").isNull().cast("int")).alias("missing_lineage"))
             .collect())
    for s in stats:
        recon.append(dict(table=table, file=s["_source_file"], raw_rows=expected.get(s["_source_file"]),
                          bronze_rows=s["n_rows"], missing_lineage=s["missing_lineage"]))

print(f"{'table':13s} {'file':45s} {'raw rows':>12s} {'bronze rows':>12s}  status")
for r in sorted(recon, key=lambda r: (r["table"], r["file"])):
    ok = r["raw_rows"] == r["bronze_rows"] and r["missing_lineage"] == 0
    r["status"] = "OK" if ok else "MISMATCH"
    print(f"{r['table']:13s} {r['file']:45s} {r['raw_rows'] or 0:>12,} {r['bronze_rows']:>12,}  {r['status']}")

n_files_expected = sum(len(v) for v in in_scope.values())
all_ok = len(recon) == n_files_expected and all(r["status"] == "OK" for r in recon)
bronze_gb = sum(p.stat().st_size for p in BRONZE_DIR.rglob("*.parquet")) / 1e9
print(f"\nFiles reconciled: {len(recon)}/{n_files_expected} | rows in Bronze: {sum(r['bronze_rows'] for r in recon):,}"
      f" | Bronze size: {bronze_gb:.2f} GB | free space left: {free_gb():.1f} GB")
print("Reconciliation:", "PASSED" if all_ok else "FAILED")

### Part 2.4: Bronze Summary

## Part 3: Silver Layer
### Part 3.1: Data understanding

### Part 3.2: Data Model

### Part 3.3: Standardization

### Part 3.4: Quality and Quarantine rules

### Part 3.5: Deduplication

### Part 3.6: Enrichment

### Part 3.7: before-and-after evidence of quality

### Part 3.8: Writing certified data and quarantine data

### Part 3.9: Performance

## Part 4: Gold Layer
### Part 4.1: Construction

### Part 4.2: Validation

### Part 4.3: Publication

## Part 5: validation summary

## Part 6: Conclusion and limits